# MultiAgent-CTDE-Lab: Getting Started Tutorial

Welcome to the MultiAgent-CTDE-Lab! This notebook will guide you through the basics of using this multi-agent reinforcement learning framework.

## Table of Contents
1. [Setup and Installation](#setup)
2. [Understanding Environments](#environments)
3. [Training MADDPG](#maddpg)
4. [Training CommMADDPG](#comm-maddpg)
5. [Visualization](#visualization)
6. [Next Steps](#next-steps)

## 1. Setup and Installation <a id='setup'></a>

First, let's import the necessary libraries and verify the setup.

In [ ]:
import sys
import os

# Add parent directory to path
sys.path.append(os.path.dirname(os.path.abspath(os.getcwd())))

import numpy as np
import torch
import matplotlib.pyplot as plt
from IPython.display import clear_output

print(f"NumPy version: {np.__version__}")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

## 2. Understanding Environments <a id='environments'></a>

Let's explore the Cooperative Navigation environment.

In [ ]:
from environments.cooperative_navigation import CooperativeNavigation

# Create environment
env = CooperativeNavigation(num_agents=3, num_landmarks=3)

# Get environment info
env_info = env.get_env_info()

print("Environment Information:")
print(f"  Number of agents: {env_info['num_agents']}")
print(f"  State dimension: {env_info['state_dim']}")
print(f"  Action dimension: {env_info['action_dim']}")
print(f"  Max steps: {env_info['max_steps']}")

### Test Environment Dynamics

In [ ]:
# Reset environment
states = env.reset()

print(f"Initial states (one per agent):")
for i, state in enumerate(states):
    print(f"  Agent {i}: shape {state.shape}")

# Take random actions
actions = [np.random.uniform(-1, 1, env_info['action_dim']) for _ in range(env_info['num_agents'])]

next_states, rewards, dones, info = env.step(actions)

print(f"\nAfter one step:")
print(f"  Rewards: {rewards}")
print(f"  Info: {info}")

### Visualize Environment

In [ ]:
# Run a random episode and visualize
states = env.reset()

for step in range(20):
    actions = [np.random.uniform(-1, 1, 2) for _ in range(3)]
    states, rewards, dones, info = env.step(actions)
    
    if all(dones):
        break

# Render final state
env.render(mode='human')
plt.show()

## 3. Training MADDPG <a id='maddpg'></a>

Now let's train a MADDPG agent on the environment.

In [ ]:
from algorithms.maddpg import MADDPG

# Create MADDPG agent
agent = MADDPG(
    num_agents=env_info['num_agents'],
    state_dim=env_info['state_dim'],
    action_dim=env_info['action_dim'],
    hidden_dim=128,
    batch_size=64,
    device='cpu'
)

print("MADDPG agent created!")
print(f"  Number of agents: {len(agent.agents)}")
print(f"  Replay buffer size: {len(agent.replay_buffer)}")

### Training Loop

In [ ]:
# Training parameters
num_episodes = 200
max_steps = 100

# Tracking
episode_rewards = []
episode_lengths = []

# Training loop
for episode in range(num_episodes):
    states = env.reset()
    episode_reward = 0
    step = 0
    
    while step < max_steps:
        # Select actions with exploration noise
        noise = max(0.05, 0.5 * (0.995 ** episode))
        actions = agent.act(states, noise=noise)
        
        # Execute actions
        global_state = env.get_global_state()
        next_states, rewards, dones, info = env.step(actions)
        next_global_state = env.get_global_state()
        
        # Store transition
        agent.step(states, actions, rewards, next_states, dones,
                  global_state, next_global_state)
        
        # Update agent
        if len(agent.replay_buffer) > agent.batch_size:
            agent.update()
        
        states = next_states
        episode_reward += sum(rewards)
        step += 1
        
        if all(dones):
            break
    
    episode_rewards.append(episode_reward)
    episode_lengths.append(step)
    
    # Print progress
    if (episode + 1) % 20 == 0:
        avg_reward = np.mean(episode_rewards[-20:])
        print(f"Episode {episode + 1}/{num_episodes} | Avg Reward: {avg_reward:.2f} | Noise: {noise:.3f}")

print("\nTraining completed!")

### Visualize Training Progress

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Plot rewards
axes[0].plot(episode_rewards, alpha=0.3, label='Raw')
window = 20
smoothed = [np.mean(episode_rewards[max(0, i-window):i+1]) for i in range(len(episode_rewards))]
axes[0].plot(smoothed, linewidth=2, label='Smoothed')
axes[0].set_xlabel('Episode')
axes[0].set_ylabel('Total Reward')
axes[0].set_title('Training Rewards')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Plot episode lengths
axes[1].plot(episode_lengths, alpha=0.6)
axes[1].set_xlabel('Episode')
axes[1].set_ylabel('Episode Length')
axes[1].set_title('Episode Lengths')
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

### Test Trained Agent

In [ ]:
# Test the trained agent
test_episodes = 5
test_rewards = []

for episode in range(test_episodes):
    states = env.reset()
    episode_reward = 0
    step = 0
    
    while step < max_steps:
        # No exploration noise during testing
        actions = agent.act(states, noise=0.0)
        states, rewards, dones, info = env.step(actions)
        
        episode_reward += sum(rewards)
        step += 1
        
        if all(dones):
            break
    
    test_rewards.append(episode_reward)
    print(f"Test Episode {episode + 1}: Reward = {episode_reward:.2f}")

print(f"\nAverage Test Reward: {np.mean(test_rewards):.2f} ± {np.std(test_rewards):.2f}")

## 4. Training CommMADDPG <a id='comm-maddpg'></a>

Let's try the communication-enhanced version!

In [ ]:
from algorithms.comm_maddpg import CommMADDPG

# Reset environment
env = CooperativeNavigation(num_agents=3, num_landmarks=3)
env_info = env.get_env_info()

# Create CommMADDPG agent with communication
comm_agent = CommMADDPG(
    num_agents=env_info['num_agents'],
    state_dim=env_info['state_dim'],
    action_dim=env_info['action_dim'],
    hidden_dim=128,
    comm_type='commnet',
    num_comm_rounds=1,
    batch_size=64,
    device='cpu'
)

print("CommMADDPG agent created with communication!")

In [ ]:
# Quick training (fewer episodes for demo)
comm_rewards = []

for episode in range(200):
    states = env.reset()
    episode_reward = 0
    step = 0
    
    while step < max_steps:
        noise = max(0.05, 0.5 * (0.995 ** episode))
        actions = comm_agent.act(states, noise=noise)
        
        global_state = env.get_global_state()
        next_states, rewards, dones, info = env.step(actions)
        next_global_state = env.get_global_state()
        
        comm_agent.step(states, actions, rewards, next_states, dones,
                       global_state, next_global_state)
        
        if len(comm_agent.replay_buffer) > comm_agent.batch_size:
            comm_agent.update()
        
        states = next_states
        episode_reward += sum(rewards)
        step += 1
        
        if all(dones):
            break
    
    comm_rewards.append(episode_reward)
    
    if (episode + 1) % 20 == 0:
        avg_reward = np.mean(comm_rewards[-20:])
        print(f"Episode {episode + 1}/200 | Avg Reward: {avg_reward:.2f}")

print("\nCommMADDPG training completed!")

### Compare MADDPG vs CommMADDPG

In [ ]:
plt.figure(figsize=(12, 5))

# Smooth both curves
window = 20
maddpg_smoothed = [np.mean(episode_rewards[max(0, i-window):i+1]) for i in range(len(episode_rewards))]
comm_smoothed = [np.mean(comm_rewards[max(0, i-window):i+1]) for i in range(len(comm_rewards))]

plt.plot(maddpg_smoothed, linewidth=2, label='MADDPG', alpha=0.8)
plt.plot(comm_smoothed, linewidth=2, label='CommMADDPG', alpha=0.8)

plt.xlabel('Episode', fontsize=12)
plt.ylabel('Average Reward', fontsize=12)
plt.title('MADDPG vs CommMADDPG Performance', fontsize=14, fontweight='bold')
plt.legend(fontsize=11)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print(f"\nFinal Performance:")
print(f"  MADDPG: {np.mean(episode_rewards[-20:]):.2f}")
print(f"  CommMADDPG: {np.mean(comm_rewards[-20:]):.2f}")

## 5. Visualization <a id='visualization'></a>

Let's visualize agent behaviors and trajectories.

In [ ]:
# Collect trajectory data
states = env.reset()
trajectories = [[] for _ in range(env_info['num_agents'])]

for step in range(50):
    # Record positions
    for i in range(env_info['num_agents']):
        trajectories[i].append(env.agent_pos[i].copy())
    
    # Take actions
    actions = comm_agent.act(states, noise=0.0)
    states, rewards, dones, info = env.step(actions)
    
    if all(dones):
        break

# Convert to numpy arrays
trajectories = [np.array(traj) for traj in trajectories]

print(f"Collected {len(trajectories)} trajectories with {len(trajectories[0])} timesteps")

In [ ]:
# Plot trajectories
from utils.visualization import TrainingVisualizer

viz = TrainingVisualizer(save_dir='notebook_viz')

viz.plot_agent_trajectories(
    trajectories,
    landmarks=env.landmark_pos,
    save_name='agent_trajectories.png'
)

# Display the saved plot
from IPython.display import Image
Image('notebook_viz/agent_trajectories.png')

## 6. Next Steps <a id='next-steps'></a>

Congratulations! You've learned the basics of using MultiAgent-CTDE-Lab. Here are some next steps:

1. **Try different environments**: Experiment with Predator-Prey
2. **Tune hyperparameters**: Adjust learning rates, hidden dimensions, etc.
3. **Longer training**: Train for more episodes to see convergence
4. **Compare algorithms**: Run systematic comparisons
5. **Implement custom environment**: Create your own multi-agent task

### Resources

- **README.md**: Project overview
- **USAGE_GUIDE.md**: Detailed usage instructions
- **EXAMPLES.md**: Quick reference examples
- **Other notebooks**: Check out the visualization and experiment comparison notebooks

Happy learning! 🚀